In [1]:
import torchxrayvision as xrv
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import Callable

# Projection algorithms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

In [2]:
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

In [3]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.widgets import RectangleSelector

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Constants and Configs

In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed seed for reproducibility
RANDOM_STATE = 42

### Load model

In [6]:
def load_model(model_weights: str = "densenet121-res224-chex"):
    model = xrv.models.DenseNet(weights=model_weights)
    model.eval()
    return model.to(DEVICE)

In [7]:
# MODEL_NAME = "densenet121-res224-all"
MODEL_NAME = "densenet121-res224-chex"
DENSENET121_RES244_CHEX_MODEL = load_model(model_weights=MODEL_NAME)
DENSENET121_RES244_CHEX_MODEL.eval()

XRV-DenseNet121-densenet121-res224-chex

In [8]:
MODEL_PATHOLOGIES = [str(path) for path in DENSENET121_RES244_CHEX_MODEL.pathologies if path != '']

In [9]:
# From the ~65 000 images we only load a subset
MAX_NUMBER_OF_DATA = 10000

### Load and Prepare DATASET

In [10]:
def load_dataset(image_path: str = './chest_data', datasetClass = xrv.datasets.CheX_Dataset):
    transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
    d_nih = datasetClass(imgpath=image_path, transform=transform, views=["PA", "AP"], unique_patients=True)
    print(f'found {len(d_nih)} images')
    return d_nih

In [11]:
#XRAY_DATASET = load_dataset('./nih_data_small', xrv.datasets.NIH_Dataset)
XRAY_DATASET = load_dataset('./chest_data', xrv.datasets.CheX_Dataset)
# We filterout pathologies which are not predicted by the model
DATASET_PATHOLOGIES = [str(path) for path in XRAY_DATASET.pathologies if path in MODEL_PATHOLOGIES]

# Align datasets to use the same pathologies
xrv.datasets.relabel_dataset(DATASET_PATHOLOGIES, XRAY_DATASET)
# Create a subset using indices
n = min(MAX_NUMBER_OF_DATA, len(XRAY_DATASET))
XRAY_DATASET = xrv.datasets.SubsetDataset(XRAY_DATASET, idxs=np.arange(0, n))
DATASET_LABEL_INDICES_MAP = {
        name: DATASET_PATHOLOGIES.index(name)
        for name in DATASET_PATHOLOGIES
}
DATASET_INDICES_LABEL_MAP = {DATASET_LABEL_INDICES_MAP[l]:l for l in DATASET_LABEL_INDICES_MAP}
print(f'Loaded {len(XRAY_DATASET)} images')
print(f'There are now {len(XRAY_DATASET.pathologies)} pathologies.')

found 64534 images
{np.str_('Pleural Other'), np.str_('Support Devices')} will be dropped
Loaded 10000 images
There are now 11 pathologies.


In [12]:
XRAY_DATASET.csv

,PatientID,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,...,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,view,patientid,age_years,sex_male,sex_female
0,patient00001,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68.0,Frontal,AP,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,AP,00001,68.0,False,True
1,patient00002,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87.0,Frontal,AP,NaN,NaN,-1.0,1.0,...,NaN,-1.0,NaN,1.0,NaN,AP,00002,87.0,False,True
2,patient00003,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41.0,Frontal,AP,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,AP,00003,41.0,True,False
3,patient00004,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20.0,Frontal,PA,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,PA,00004,20.0,False,True
4,patient00005,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33.0,Frontal,PA,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,PA,00005,33.0,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,patient09998,CheXpert-v1.0-small/train/patient09998/study1/...,Female,88.0,Frontal,AP,NaN,0.0,NaN,NaN,...,0.0,0.0,NaN,NaN,-1.0,AP,09998,88.0,False,True
9996,patient09999,CheXpert-v1.0-small/train/patient09999/study1/...,Female,50.0,Frontal,AP,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,AP,09999,50.0,False,True
9997,patient10000,CheXpert-v1.0-small/train/patient10000/study1/...,Male,64.0,Frontal,AP,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,AP,10000,64.0,True,False
9998,patient10001,CheXpert-v1.0-small/train/patient10001/study2/...,Female,30.0,Frontal,AP,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,AP,10001,30.0,False,True


In [13]:
def get_patient_gender(dataset):
    metadata = getattr(dataset, "csv", None)

    def normalize_gender(value):
        value = str(value).strip().lower()

        if value == 'm' or value == 'male':
            return 0.0
        if value == 'f' or value == 'female':
            return 1.0
        return 2.0
    metadata_name = "Sex" if "Sex" in metadata else "Patient Gender"
    return metadata[metadata_name].map(normalize_gender).to_numpy()
    #return metadata["Patient Gender"].map(normalize_gender).to_numpy()

In [14]:
XRAY_GENDER_LIST = get_patient_gender(XRAY_DATASET)

In [15]:
def get_patient_age(dataset):
    metadata = getattr(dataset, "csv", None)

    def parse_age(value):
        value = str(value).strip().lower()

        try:
            return int(float(value))
        except ValueError:
            # TODO: handle proberly
            return -1
    metadata_name = "Age" if "Age" in metadata else "Patient Age"
    return metadata[metadata_name].map(parse_age).to_numpy()
    #return metadata["Patient Age"].map(parse_age).to_numpy()

In [16]:
XRAY_AGE_LIST = get_patient_age(XRAY_DATASET)

In [17]:
def get_dataset_views(dataset):
    metadata = getattr(dataset, "csv", None)

    def parse_age(value):
        value = str(value).strip().lower()

        try:
            return value
        except ValueError:
            # TODO: handle proberly
            return -1
    metadata_name = "AP/PA" if "AP/PA" in metadata else "View Position"
    return metadata[metadata_name].map(parse_age).to_numpy()

In [18]:
XRAY_VIEW_LIST = get_dataset_views(XRAY_DATASET)

In [19]:
def create_image_feature_space_and_label(dataset):

    source_indices = []
    target_indices = []

    for org_index, path in enumerate(XRAY_DATASET.pathologies):
        if path in DATASET_LABEL_INDICES_MAP:
            source_indices.append(org_index)
            target_indices.append(DATASET_LABEL_INDICES_MAP[path])

    source_indices = np.asarray(source_indices)
    target_indices = np.asarray(target_indices)


    image_features = []
    labels = []

    n_labels = len(DATASET_PATHOLOGIES)

    for i in range(len(dataset)):
        sample = dataset[i]

        # Image
        img = np.asarray(sample["img"], dtype=np.float32).squeeze()
        image_features.append(img)

        # Labels
        label = np.nan_to_num(
            np.asarray(sample["lab"]).reshape(-1)
        )

        adapted = np.zeros(n_labels, dtype=label.dtype)
        adapted[target_indices] = label[source_indices]

        labels.append(adapted)

    return np.asarray(image_features, dtype=np.float32), np.stack(labels)

In [20]:
XRAY_IMAGE_FEATURES, XRAY_LABELS_LIST = create_image_feature_space_and_label(XRAY_DATASET)

In [21]:
def get_model_to_dataset_indices(model = DENSENET121_RES244_CHEX_MODEL, dataset_label_indices_map=DATASET_LABEL_INDICES_MAP):
    return {
        model_idx: dataset_label_indices_map[pathology]
        for model_idx, pathology in enumerate(model.pathologies)
        if pathology in dataset_label_indices_map
    }   

In [22]:
MODEL_TO_DATASET_INDICES_MAP = get_model_to_dataset_indices()
DATASET_TO_MODEL_INDICES_MAP = {MODEL_TO_DATASET_INDICES_MAP[key]:key for key in MODEL_TO_DATASET_INDICES_MAP}

### Prediction of the model

In [23]:
def convert_to_dataset_label_list(model_prediction_score, dataset_labels_list = XRAY_LABELS_LIST,
                                  dataset_to_model_indices_map=DATASET_TO_MODEL_INDICES_MAP
                                  ):
    new_prection_list = np.zeros_like(dataset_labels_list)
    for selectedDatasetIndex in dataset_to_model_indices_map:
        selectedModelIndex = dataset_to_model_indices_map[selectedDatasetIndex]
        new_prection_list[:, selectedDatasetIndex] = model_prediction_score[:, selectedModelIndex]
    return new_prection_list

In [24]:
def calculate_model_predictions_score(
    feature_space,
    model,
    batch_size=32
):
    model.eval()

    # Convert flattened images to (N, 1, 224, 224)
    images = torch.as_tensor(feature_space, dtype=torch.float32)
    images = images.reshape(-1, 1, 224, 224)

    # Run inference in batches
    all_outputs = []

    with torch.no_grad():
        for start_idx in range(0, len(images), batch_size):
            end_idx = start_idx + batch_size

            batch = images[start_idx:end_idx].to(DEVICE)

            batch_outputs = model(batch)

            # Move each batch back to CPU immediately
            batch_outputs = batch_outputs.cpu().numpy()


            all_outputs.append(batch_outputs)

    # Combine all batches
    selected_outputs = np.concatenate(all_outputs, axis=0)
    return selected_outputs

In [25]:
MODEL_SCORE_PREDICTIONS_CONVERTED = convert_to_dataset_label_list(calculate_model_predictions_score(XRAY_IMAGE_FEATURES,
                                                       DENSENET121_RES244_CHEX_MODEL))

In [26]:
def evaluate_multilabel_model(
    y_true,
    y_scores,
    pathology_names,
    threshold=0.7
):
    """
    Evaluate a multi-label classification model.

    Creates ONE interactive Plotly figure containing a 3-column
    subgroup for every pathology:

        [Confusion Matrix] [ROC Curve] [Precision-Recall Curve]

    Each pathology gets its own row.

    Parameters
    ----------
    y_true : array-like, shape (n_samples, n_labels)
        Ground-truth binary labels.

    y_scores : array-like, shape (n_samples, n_labels)
        Model probabilities/scores.

    pathology_names : list[str]
        Names of the pathologies.

    threshold : float, default=0.7
        Threshold used to convert scores into binary predictions.

    Returns
    -------
    results : dict
        Dictionary containing metrics and the final Plotly figure.
    """

    # ---------------------------------------------------------
    # Validate input
    # ---------------------------------------------------------

    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    assert y_true.shape == y_scores.shape, (
        f"Shape mismatch: {y_true.shape} vs {y_scores.shape}"
    )

    assert y_true.ndim == 2, (
        f"Expected 2D arrays, got {y_true.ndim}D"
    )

    assert y_true.shape[1] == len(pathology_names), (
        f"Number of columns ({y_true.shape[1]}) does not match "
        f"number of pathology names ({len(pathology_names)})"
    )

    n_pathologies = len(pathology_names)

    # ---------------------------------------------------------
    # Create ONE figure
    # ---------------------------------------------------------

    fig = make_subplots(
        rows=n_pathologies,
        cols=3,

        column_titles=[
            "Confusion Matrix",
            "ROC Curve",
            "Precision–Recall Curve"
        ],

        row_titles=pathology_names,

        horizontal_spacing=0.045,
        vertical_spacing=0.035,

        specs=[
            [
                {"type": "heatmap"},
                {"type": "xy"},
                {"type": "xy"}
            ]
            for _ in range(n_pathologies)
        ]
    )

    results = {}

    # ---------------------------------------------------------
    # Evaluate each pathology
    # ---------------------------------------------------------

    for i, pathology in enumerate(pathology_names):

        row = i + 1

        true = y_true[:, i]
        scores = y_scores[:, i]

        # -----------------------------------------------------
        # Binary predictions
        # -----------------------------------------------------

        pred = (scores >= threshold).astype(int)

        # -----------------------------------------------------
        # Confusion matrix
        # -----------------------------------------------------

        cm = confusion_matrix(
            true,
            pred,
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        # -----------------------------------------------------
        # ROC + Precision-Recall
        # -----------------------------------------------------

        if len(np.unique(true)) == 2:

            # ROC
            fpr, tpr, roc_thresholds = roc_curve(
                true,
                scores
            )

            roc_auc = auc(
                fpr,
                tpr
            )

            # Precision-Recall
            precision, recall, pr_thresholds = (
                precision_recall_curve(
                    true,
                    scores
                )
            )

            ap = average_precision_score(
                true,
                scores
            )

            prevalence = np.mean(true)

        else:

            fpr = None
            tpr = None
            roc_thresholds = None
            roc_auc = np.nan

            precision = None
            recall = None
            pr_thresholds = None
            ap = np.nan

            prevalence = np.mean(true)

        # -----------------------------------------------------
        # Store metrics
        # -----------------------------------------------------

        results[pathology] = {
            "confusion_matrix": cm,

            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,

            "fpr": fpr,
            "tpr": tpr,
            "roc_thresholds": roc_thresholds,
            "roc_auc": roc_auc,

            "precision": precision,
            "recall": recall,
            "pr_thresholds": pr_thresholds,
            "average_precision": ap,

            "prevalence": prevalence
        }

        # =====================================================
        # 1. CONFUSION MATRIX
        # =====================================================

        fig.add_trace(
            go.Heatmap(
                z=cm,

                x=[
                    "Predicted Negative",
                    "Predicted Positive"
                ],

                y=[
                    "Actual Negative",
                    "Actual Positive"
                ],

                colorscale="Blues",
                showscale=False,

                text=cm,
                texttemplate="%{text}",

                hovertemplate=(
                    f"<b>{pathology}</b><br>"
                    "Actual: %{y}<br>"
                    "Predicted: %{x}<br>"
                    "Count: %{z}<extra></extra>"
                )
            ),
            row=row,
            col=1
        )

        # =====================================================
        # 2. ROC CURVE
        # =====================================================

        if fpr is not None:

            fig.add_trace(
                go.Scatter(
                    x=fpr,
                    y=tpr,

                    mode="lines",

                    name=f"{pathology} ROC",

                    line=dict(width=3),

                    hovertemplate=(
                        "FPR: %{x:.3f}<br>"
                        "TPR: %{y:.3f}<extra></extra>"
                    ),

                    showlegend=False
                ),
                row=row,
                col=2
            )

            # Random baseline
            fig.add_trace(
                go.Scatter(
                    x=[0, 1],
                    y=[0, 1],

                    mode="lines",

                    line=dict(
                        dash="dash",
                        color="gray"
                    ),

                    hoverinfo="skip",
                    showlegend=False
                ),
                row=row,
                col=2
            )

        # =====================================================
        # 3. PRECISION-RECALL CURVE
        # =====================================================

        if precision is not None:

            fig.add_trace(
                go.Scatter(
                    x=recall,
                    y=precision,

                    mode="lines",

                    name=f"{pathology} PR",

                    line=dict(width=3),

                    hovertemplate=(
                        "Recall: %{x:.3f}<br>"
                        "Precision: %{y:.3f}<extra></extra>"
                    ),

                    showlegend=False
                ),
                row=row,
                col=3
            )

            # Prevalence baseline
            fig.add_hline(
                y=prevalence,

                line_dash="dash",
                line_color="gray",

                annotation_text=(
                    f"Prevalence = {prevalence:.3f}"
                ),

                annotation_position="top right",

                row=row,
                col=3
            )

    # =========================================================
    # AXES
    # =========================================================

    for row in range(1, n_pathologies + 1):

        # -----------------------------
        # Confusion matrix
        # -----------------------------

        fig.update_xaxes(
            title_text="Predicted",
            row=row,
            col=1
        )

        fig.update_yaxes(
            title_text="Actual",
            autorange="reversed",
            row=row,
            col=1
        )

        # -----------------------------
        # ROC
        # -----------------------------

        fig.update_xaxes(
            title_text="False Positive Rate",
            range=[0, 1],
            row=row,
            col=2
        )

        fig.update_yaxes(
            title_text="True Positive Rate",
            range=[0, 1],
            row=row,
            col=2
        )

        # -----------------------------
        # Precision-Recall
        # -----------------------------

        fig.update_xaxes(
            title_text="Recall",
            range=[0, 1],
            row=row,
            col=3
        )

        fig.update_yaxes(
            title_text="Precision",
            range=[0, 1],
            row=row,
            col=3
        )

    # =========================================================
    # Add metric summaries to row titles
    # =========================================================

    for i, pathology in enumerate(pathology_names):

        metrics = results[pathology]

        auc_value = metrics["roc_auc"]
        ap_value = metrics["average_precision"]

        if np.isnan(auc_value):
            auc_text = "AUC = N/A"
        else:
            auc_text = f"AUC = {auc_value:.3f}"

        if np.isnan(ap_value):
            ap_text = "AP = N/A"
        else:
            ap_text = f"AP = {ap_value:.3f}"

        # Put metric information into the subplot title
        fig.add_annotation(
            text=(
                f"<b>{pathology}</b>"
                f"<br><sup>{auc_text} | {ap_text}</sup>"
            ),

            x=-0.10,
            y=1 - (
                i + 0.5
            ) / n_pathologies,

            xref="paper",
            yref="paper",

            showarrow=False,

            xanchor="right",
            yanchor="middle",

            align="right"
        )

    # =========================================================
    # Layout
    # =========================================================

    fig.update_layout(

        title=dict(
            text=(
                "Multi-Label Chest X-Ray Model Evaluation"
                f"<br><sup>Classification threshold = {threshold}</sup>"
            ),
            x=0.5
        ),

        template="plotly_white",

        # More vertical space per pathology
        height=max(650, 420 * n_pathologies),

        # Slightly wider figure
        width=1500,

        margin=dict(
            l=150,
            r=40,
            t=100,
            b=50
        ),

        showlegend=False
    )

    # =========================================================
    # Display ONE figure
    # =========================================================

    fig.show()

    # Store the complete figure
    results["figure"] = fig

    return results

In [27]:
results = evaluate_multilabel_model(
    y_true=XRAY_LABELS_LIST,
    y_scores=MODEL_SCORE_PREDICTIONS_CONVERTED,
    pathology_names=DATASET_PATHOLOGIES,
    threshold=0.7
)

### Downprojection

In [28]:
class Downprojector:

    def __init__(self, image_features):
        self.projection_cache = {}
        self.image_features = image_features


    def get_feature_matrix(self, feature_space):
        return self.image_features

    """
    PCA is used as a preprocessing step for the non-linear
        projection techniques.
    """
    def prepare_pca_features(self, feature_space):
        cache_key = f"{feature_space}_pca"
         
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]
         
        X = self.get_feature_matrix(feature_space)
         
             # Keep at most 50 dimensions before expensive
             # non-linear dimensionality reduction.
        n_components = min(
                 50,
                 X.shape[0],
                 X.shape[1]
             )
         
        pca = PCA(
                 n_components=n_components,
                 random_state=RANDOM_STATE
        )
         
        X_pca = pca.fit_transform(X)
         
        self.projection_cache[cache_key] = {
                 "data": X_pca,
                 "pca": pca
             }
         
        return self.projection_cache[cache_key]

    def calculate_projection(self, feature_space: str, method: str,
                             target_labels=None,
                             label_name=None,
                             dataset_label_to_index_map = DATASET_LABEL_INDICES_MAP,):
        
        cache_key = (
                f"{feature_space}_{method}"
            ) if label_name is None else f"{feature_space}_{method}_{label_name}"
        
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]


        if method == "PCA":

            X = self.get_feature_matrix(feature_space)

            reducer = PCA(
                n_components=2,
                random_state=RANDOM_STATE
            )

            embedding = reducer.fit_transform(X)
        elif method == "t-SNE":
        
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]
        
                # t-SNE requires perplexity < number of samples.
            perplexity = min(
                    30,
                    max(5, (X_pca.shape[0] - 1) // 3)
                )
        
            reducer = TSNE(
                    n_components=2,
                    perplexity=perplexity,
                    init="pca",
                    learning_rate="auto",
                    random_state=RANDOM_STATE
                )
        
            embedding = reducer.fit_transform(
                    X_pca
                )
        elif method == "UMAP":
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]

            n_neighbors = min(
                    15,
                    max(2, X_pca.shape[0] - 1)
                )

            reducer = umap.UMAP(
                    n_components=2,
                    n_neighbors=n_neighbors,
                    min_dist=0.1,
                    metric="euclidean",
                    random_state=RANDOM_STATE
                )

            embedding = reducer.fit_transform(
                    X_pca
                )
        # Supervised UMAP
        elif method == "S-UMAP":

            if target_labels is None:
                raise ValueError('Supervised UMAP needs target labels')

            if label_name is None:
                raise ValueError('Supervised UMAP needs a label name / pathologie')
            
            X_pca = self.prepare_pca_features(
                                feature_space
                            )["data"]
            
            n_neighbors = min(
                                15,
                                max(2, X_pca.shape[0] - 1)
                            )
            
            reducer = umap.UMAP(
                                n_components=2,
                                n_neighbors=n_neighbors,
                                min_dist=0.1,
                                metric="euclidean",
                                random_state=RANDOM_STATE
                            )

            target_index = dataset_label_to_index_map[label_name]

            Y = target_labels[:, target_index]
            
            embedding = reducer.fit_transform(
                                X_pca,
                                Y
                            )
        else:
            raise ValueError(
                f"Unknown projection method: {method}"
                )
        self.projection_cache[cache_key] = embedding
        return embedding

### Code for the generic viewer

In [29]:
@dataclass
class XRayVisualizationData:

    feature_space: np.ndarray
    projection_method: str

    model_score_predictions: np.ndarray
    ground_truth: np.ndarray
    dataset_label_indices_map: dict

    dataset_gender_list: np.ndarray
    dataset_ages: np.ndarray

    selected_pathology: str
    threshold: float
    y_true: np.ndarray = None
    y_pred: np.ndarray = None
    y_scores: np.ndarray = None

    def __post_init__(self):
        label_idx = self.dataset_label_indices_map[self.selected_pathology]

        # Extract ground truth and prediction scores
        self.y_true = np.asarray(self.ground_truth[:, label_idx])
       
        
        self.y_scores = np.asarray(self.model_score_predictions[:, label_idx])

        self.y_pred = (self.y_scores >= self.threshold).astype(int)

In [30]:
VisualizationFunction = Callable[
    [XRayVisualizationData],
    None
]

In [31]:
def create_hover_template(data: XRayVisualizationData, mask):
    
    customdata=np.column_stack((
                    data.y_true[mask],
                    data.y_scores[mask],
                    data.y_pred[mask],
                    np.asarray(data.dataset_ages)[mask],
                    np.asarray(data.dataset_gender_list)[mask]
                ))

    hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Component 1: %{x:.3f}<br>"
                    "Component 2: %{y:.3f}<br>"
                    "Ground truth: %{customdata[0]}<br>"
                    "Prediction score: %{customdata[1]:.3f}<br>"
                    "Binary prediction: %{customdata[2]}<br>"
                    "Age: %{customdata[3]} years<br>"
                    "Gender: %{customdata[4]}"
                    "<extra></extra>"
                )
    return customdata, hovertemplate

In [32]:
def create_viewer(downprojector: Downprojector, visualization_fn: VisualizationFunction):

    plot_output = widgets.Output()

    progress_output = widgets.Output()

    plot_button = widgets.Button(
        description="Create selection plot",
        button_style="primary",
        icon="refresh"
    )

    pathology_dropdown = widgets.Dropdown(
        options=DATASET_PATHOLOGIES,
        value=DATASET_PATHOLOGIES[0],
        description="Pathology:",
        layout=widgets.Layout(width="350px")
    )

    projection_dropdown = widgets.Dropdown(
        options=[
            "PCA",
            "t-SNE",
            "UMAP",
            "S-UMAP"
        ],
        value="PCA",
        description="Projection:",
        layout=widgets.Layout(width="300px")
    )

    threshold_input = widgets.FloatSlider(
        value=0.7,
        min=0.0,
        max=1.0,
        step=0.1,
        description='Threshold',
        disabled=False
    )

    def create_selection_plot(_=None):

        with progress_output:
            clear_output(wait=True)
            pathology = pathology_dropdown.value
            projection_method = projection_dropdown.value

            target_labels = None
            label_name = None
            threshold = float(threshold_input.value)
            
            if projection_method == 'S-UMAP':
                target_labels = XRAY_LABELS_LIST
                label_name = pathology

            # Calculate the requested projection if it has not already
            # been calculated.
            embedding = downprojector.calculate_projection(
                "feature",
                projection_method,
                target_labels=target_labels,
                label_name = label_name
            )
        with plot_output:
            clear_output(wait=True)
     
            xray_data = XRayVisualizationData(
                feature_space=embedding,
                projection_method=projection_method,
                model_score_predictions=MODEL_SCORE_PREDICTIONS_CONVERTED,
                ground_truth=XRAY_LABELS_LIST,
                dataset_label_indices_map=DATASET_LABEL_INDICES_MAP,
                dataset_gender_list=XRAY_GENDER_LIST,
                dataset_ages=XRAY_AGE_LIST,
                selected_pathology=pathology,
                threshold = threshold
                )
            visualization_fn(xray_data)
            

    plot_button.on_click(create_selection_plot)

    display(
    widgets.VBox([
        widgets.HBox([
            projection_dropdown
        ]),
        widgets.HBox([
            pathology_dropdown
        ]),
        threshold_input,
        plot_button,
        progress_output,
        plot_output,
    ])
)

### Visualization functions

In [33]:
def visualize_space_traces(
    data: XRayVisualizationData,
): 
    label_index = data.dataset_label_indices_map[data.selected_pathology]

    y_true = np.asarray(data.ground_truth[:, label_index])
    gender = np.asarray(data.dataset_gender_list)

    # Define groups
    categories = {
        "Finding present · Male": (y_true == 1) & (gender == 0),
        "Finding present · Female": (y_true == 1) & (gender == 1),
        "Finding absent · Male": (y_true == 0) & (gender == 0),
        "Finding absent · Female": (y_true == 0) & (gender == 1)
    }

    # Colors represent pathology status
    colors = {
        "Finding present": "#D55E00",
        "Finding absent": "#0072B2"
    }

    # Shapes represent gender
    symbols = {
        "Male": "circle",
        "Female": "triangle-up"
    }

    traces = []
    for category, mask in categories.items():

        status, sex = category.split(" · ")

        customdata, hovertemplate = create_hover_template(data, mask)
        
        traces.append(
            go.Scatter(
                x=data.feature_space[mask, 0],
                y=data.feature_space[mask, 1],
                mode="markers",
                name=category,
                marker=dict(
                    color=colors[status],
                    symbol=symbols[sex],
                    size=7,
                    opacity=0.75 if status == "Finding present" else 0.25
                ),
                customdata=customdata,
                hovertemplate=hovertemplate
            )
        )
    
    return traces

In [34]:
def visualize_space(
    data: XRayVisualizationData,
): 
    projection_method = data.projection_method
    
    fig = go.Figure()
    traces = visualize_space_traces(data)
    fig.add_traces(traces)
    fig.update_layout(
        title=dict(
            text=f"Distribution of Chest X-ray Embeddings by {data.selected_pathology} and Gender",
            x=0.5
        ),
        xaxis_title=f"{projection_method} 1",
        yaxis_title=f"{projection_method} 2",
        template="plotly_white",
        width=1000,
        height=750,
        legend=dict(
            title="Pathology Status and Gender",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        hovermode="closest"
    )
    
    fig.show()

In [35]:
downprojector = Downprojector(XRAY_IMAGE_FEATURES.reshape(-1, 224* 224))

In [36]:
create_viewer(downprojector, visualize_space)

### Visualize predictions


In [37]:
def visualize_space_prediction_scores_traces(data: XRayVisualizationData):

    label_index = data.dataset_label_indices_map[data.selected_pathology]
      
    y_true = np.asarray(data.ground_truth[:, label_index])
    gender = np.asarray(data.dataset_gender_list)
      
          # Define groups
    categories = {
              "Finding present · Male": (y_true == 1) & (gender == 0),
              "Finding present · Female": (y_true == 1) & (gender == 1),
              "Finding absent · Male": (y_true == 0) & (gender == 0),
              "Finding absent · Female": (y_true == 0) & (gender == 1)
      }
      
    # Shapes represent gender
    symbols = {
              "Male": "circle",
              "Female": "triangle-up"
      }
      
    traces = []
      
    for i, (category, mask) in enumerate(categories.items()):
      
        status, sex = category.split(" · ")
      
        customdata, hovertemplate = create_hover_template(data, mask)
  
        scores = data.y_scores[mask]
              
        traces.append(
                  go.Scatter(
                      x=data.feature_space[mask, 0],
                      y=data.feature_space[mask, 1],
                      mode="markers",
                      name=category,
                      marker=dict(
                          color=scores,
                          colorscale="Viridis",
                          cmin=0,
                          cmax=1,
                          showscale=(i == 0),  # Show only one color bar
                          colorbar=dict(
                              title="Model score",
                              thickness=18,
                              len=0.75
                          ) if i == 0 else None,
                          symbol=symbols[sex],
                          size=7,
                          opacity=0.75 if status == "Finding present" else 0.25
                      ),
                      customdata=customdata,
                      hovertemplate=hovertemplate
                  )
              )
      

    return traces

In [38]:
def visualize_space_prediction_scores(data: XRayVisualizationData):
    projection_method = data.projection_method

    fig = go.Figure()
    traces = visualize_space_prediction_scores_traces(data)
    fig.add_traces(traces)
    
    fig.update_layout(
        title=dict(
            text=f"{projection_method} Projection Colored by {data.selected_pathology} Model Score",
            x=0.5
        ),
        xaxis_title=f"{projection_method} 1",
        yaxis_title=f"{projection_method} 2",
        template="plotly_white",
        width=1000,
        height=750,
        legend=dict(
            title="Pathology Status and Gender",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        hovermode="closest"
    )
    
    fig.show()

In [39]:
create_viewer(downprojector, visualize_space_prediction_scores)

### Compare Prediction with Dataset

### Visualize true and wrong predictions

In [40]:
def visualize_classification_outcome_traces(
    data: XRayVisualizationData
):
        label_idx = data.dataset_label_indices_map[data.selected_pathology]
    
        # Extract ground truth and prediction scores
        y_true = data.y_true
        y_scores = np.asarray(data.model_score_predictions[:, label_idx])
    
    
        y_pred = (y_scores >= data.threshold).astype(int)
    
        categories = {
            "True Positive": (y_true == 1) & (y_pred == 1),
            "True Negative": (y_true == 0) & (y_pred == 0),
            "False Positive": (y_true == 0) & (y_pred == 1),
            "False Negative": (y_true == 1) & (y_pred == 0)
        }
    
        colors = {
            "True Positive": "#009E73",
            "True Negative": "#0072B2",
            "False Positive": "#D55E00",
            "False Negative": "#CC79A7"
        }
    
        traces = []
    
        for category, mask in categories.items():
    
            customdata , hovertemplate = create_hover_template(data, mask)
    
            traces.append(
                go.Scatter(
                    x=data.feature_space[mask, 0],
                    y=data.feature_space[mask, 1],
                    mode="markers",
                    name=category,
                    marker=dict(
                        color=colors[category],
                        size=7 if "False" not in category else 9,
                        opacity=0.8,
                        line=dict(
                            width=0.5,
                            color=colors[category]
                        )
                    ),
                    customdata=customdata,
    
                    hovertemplate=hovertemplate
                )
            )
    
        return traces

In [41]:
def visualize_classification_outcome(
    data: XRayVisualizationData
):

    projection_method = data.projection_method

   
    fig = go.Figure()
    traces = visualize_classification_outcome_traces(data)
    fig.add_traces(traces)

    fig.update_layout(
        title=dict(
            text=f"Classification Outcomes in {projection_method} Embedding Space – {data.selected_pathology}",
            x=0.5
        ),
        xaxis_title=f"{projection_method} 1",
        yaxis_title=f"{projection_method} 2",
        template="plotly_white",
        width=1000,
        height=750,
        legend=dict(
            title="Classification Outcome",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        hovermode="closest",
    )

    fig.show()

In [42]:
create_viewer(downprojector, visualize_classification_outcome)

### Visualize different Metadata against each other

In [43]:
def visualize_relationship_traces(
    data: XRayVisualizationData,
    x_values,
    y_values,
    x_label="X",
    y_label="Y",
    color_values=None,
    color_label=None,
    title=None
):
        traces = []
    
        traces.append(
            go.Histogram2d(
                x=x_values,
                y=y_values,
                histnorm=None,
                colorscale="Viridis",
                colorbar=dict(
                    title="Count"
                ),
                hovertemplate=(
                    f"{x_label}: %{{x}}<br>"
                    f"{y_label}: %{{y}}<br>"
                    "Count: %{z}"
                    "<extra></extra>"
                ),
                nbinsx=30,
                nbinsy=30
            )
        )
    
        return traces

In [44]:
def visualize_relationship(
    data: XRayVisualizationData,
    x_values,
    y_values,
    x_label="X",
    y_label="Y",
    color_values=None,
    color_label=None,
    title=None
):
    fig = go.Figure()

    traces = visualize_relationship_traces(data, x_values,
                                           y_values, x_label, y_label, color_values, color_label, title)
    fig.add_traces(traces)
    fig.update_layout(
        title=title or f"{y_label} vs. {x_label}",
        xaxis_title=x_label,
        yaxis_title=y_label,
        template="plotly_white",
        width=1000,
        height=750
    )

    fig.show()

In [45]:
def visualize_age_against_score_traces(data: XRayVisualizationData):
    return visualize_relationship_traces(
        data=data,
        x_values=data.dataset_ages,
        y_values=data.y_scores,
        x_label="Patient Age",
        y_label="Model Confidence",
        title=f"Model Confidence vs. Age – {data.selected_pathology}"
    )

In [46]:
def visualize_age_against_score(data: XRayVisualizationData):
    visualize_relationship(
    data=data,
    x_values=data.dataset_ages,
    y_values=data.y_scores,
    x_label="Patient Age",
    y_label="Model Confidence",
    title=f"Model Confidence vs. Age – {data.selected_pathology}"
)

In [47]:
create_viewer(downprojector, visualize_age_against_score)

### Plot subplots

In [48]:
from plotly.subplots import make_subplots


def create_multi_pathology_viewer(
    downprojector: Downprojector,
    projection_method: str,
    visualization_fns: list[VisualizationFunction],
    visualization_names: list[str],
    model_score_predictions,
    ground_truth,
    dataset_label_indices_map,
    dataset_gender_list=None,
    dataset_ages=None,
    pathologies=DATASET_PATHOLOGIES,
    threshold=0.7
):

    n_rows = len(pathologies)
    n_cols = len(visualization_fns)

    # Create subplot titles
    subplot_titles = [
        f"{pathology} – {name}"
        for pathology in pathologies
        for name in visualization_names
    ]
    print(subplot_titles)

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.06,
        vertical_spacing=0.04,
        row_heights=[1 / n_rows] * n_rows,
        column_widths=[1 / n_cols] * n_cols
    )

    # Calculate embeddings
    # For ordinary projections, the embedding is shared by all pathologies.
    shared_embedding = None

    if projection_method != "S-UMAP":
        shared_embedding = downprojector.calculate_projection(
            "feature",
            projection_method
        )

    for row, pathology in enumerate(pathologies, start=1):

        # S-UMAP is supervised, so calculate a separate embedding
        # for each pathology.
        if projection_method == "S-UMAP":
            embedding = downprojector.calculate_projection(
                "feature",
                projection_method,
                target_labels=ground_truth,
                label_name=pathology
            )
        else:
            embedding = shared_embedding

        # Create the data object for this pathology
        xray_data = XRayVisualizationData(
            feature_space=embedding,
            projection_method=projection_method,
            model_score_predictions=model_score_predictions,
            ground_truth=ground_truth,
            dataset_label_indices_map=dataset_label_indices_map,
            dataset_gender_list=dataset_gender_list,
            dataset_ages=dataset_ages,
            selected_pathology=pathology,
            threshold=threshold
        )

        # Generate each visualization in its corresponding column
        for col, visualization_fn in enumerate(visualization_fns, start=1):

            traces = visualization_fn(xray_data)
    
            for trace in traces:
                # Avoid repeating the same legend entries for every pathology
                trace.legendgroup = trace.name
                trace.showlegend = (row == 1)

                fig.add_trace(
                    trace,
                    row=row,
                    col=col
                )
    fig.update_layout(
        title=dict(
            text=f"{projection_method} – Comparison Across Pathologies",
            x=0.5,
            y=0.99
        ),
        template="plotly_white",

        # Figure dimensions
        height=max(400 * n_rows, 800),
        width=max(700 * n_cols, 1400),

        # Reduce unused space
        margin=dict(
            l=50,
            r=120,
            t=90,
            b=50
        ),

        hovermode="closest",

        # Legend
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.01
        )
    )

    fig.update_xaxes(
        showgrid=True,
        zeroline=False,
        automargin=True
    )

    fig.update_yaxes(
        showgrid=True,
        zeroline=False,
        automargin=True
    )

    fig.show()

    return fig

In [ ]:
visualization_functions = [
    visualize_space_prediction_scores_traces,
    visualize_classification_outcome_traces,
    visualize_age_against_score_traces
]

visualization_names = [
    "Model Confidence",
    "Classification outcomes",
    "Age vs. model confidence"
]
# This creates heavy plot, execute this function only if really needed.
execute_multi_plot = False
if execute_multi_plot:
    create_multi_pathology_viewer(
        downprojector=downprojector,
        projection_method="S-UMAP",
        visualization_fns=visualization_functions,
        visualization_names= visualization_names,
        model_score_predictions=MODEL_SCORE_PREDICTIONS_CONVERTED,
        ground_truth=XRAY_LABELS_LIST,
        dataset_label_indices_map=DATASET_LABEL_INDICES_MAP,
        dataset_gender_list=XRAY_GENDER_LIST,
        dataset_ages=XRAY_AGE_LIST,
        threshold=0.7
    )